# 01. First look at the processed GEO data (GSE222897)

Goal: see which files GEO gives us, which samples they cover, and what the count tables look like.

In [ ]:
from pathlib import Path
import gzip
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'image.cmap': 'plasma',
    'figure.dpi': 120,
})
pd.set_option('display.max_columns', 50)

raw = Path('../data/raw')

## 1. Which files did we download?

In [ ]:
for p in sorted(raw.iterdir()):
    print(f'{p.stat().st_size / 1e6:10.2f} MB  {p.name}')

## 2. Which samples are there?

GEO's family file lists every sample with its title and description. This tells us which columns belong to which cell line and replicate.

In [ ]:
rows = []
current = None
with gzip.open(raw / 'GSE222897_family.soft.gz', 'rt') as f:
    for line in f:
        if line.startswith('^SAMPLE'):
            current = {'gsm': line.split('=')[1].strip()}
            rows.append(current)
        elif current is not None and line.startswith('!Sample_title'):
            current['title'] = line.split('=', 1)[1].strip()
        elif current is not None and line.startswith('!Sample_characteristics_ch1'):
            key, _, value = line.split('=', 1)[1].strip().partition(': ')
            current[key] = value

samples = pd.DataFrame(rows)
print(len(samples), 'samples')
samples

## 3. Peek at each table

`sep=None` lets pandas guess whether the file is comma- or tab-separated.

In [ ]:
tables = {}
for p in sorted(raw.glob('GSE222897_*')):
    if p.name.endswith('soft.gz'):
        continue
    try:
        tables[p.name] = pd.read_csv(p, sep=None, engine='python')
        print(p.name, tables[p.name].shape)
    except Exception as e:
        print(p.name, 'could not be read as a table:', e)

In [ ]:
for name, df in tables.items():
    print('\n' + name)
    display(df.head())

## Next

- Identify which table is the HCDR3 count matrix and which is the DESeq2 output.
- Check the replicate columns match the sample list above.
- Plot clone abundance distributions per replicate.